# RL Routing — Policy Comparison Analysis

Loads `experiments/results/evaluation_results.json` and produces four figures:
1. Accuracy vs workload (auto-approval rate) tradeoff
2. Per-difficulty-tier error rates (grouped bar)
3. Routing action distribution per policy (stacked bar)
4. Learning curve from TensorBoard monitor logs

In [ ]:
import json
import os
from pathlib import Path

import matplotlib
import matplotlib.pyplot as plt
import numpy as np

matplotlib.rcParams.update({
    'font.family': 'serif',
    'font.size': 11,
    'axes.spines.top': False,
    'axes.spines.right': False,
    'axes.grid': True,
    'grid.alpha': 0.3,
    'figure.dpi': 150,
})

REPO_ROOT = Path('../../')
RESULTS_DIR = REPO_ROOT / 'experiments' / 'results'
FIGURES_DIR = RESULTS_DIR / 'figures'
FIGURES_DIR.mkdir(parents=True, exist_ok=True)

with open(RESULTS_DIR / 'evaluation_results.json') as f:
    data = json.load(f)

policies = data['policies']
names = [p['policy'] for p in policies]
short_names = ['Baseline', 'PPO-A', 'PPO-B', 'PPO-C']

print(f'Loaded {len(policies)} policies, {data["n_transactions"]} transactions')

## Figure 1: Accuracy vs Workload Tradeoff

In [ ]:
fig, ax = plt.subplots(figsize=(7, 5))

colors = ['#555555', '#2196F3', '#4CAF50', '#FF5722']
markers = ['s', 'o', '^', 'D']

for p, name, color, marker in zip(policies, short_names, colors, markers):
    x = p['auto_approval_rate']        # workload proxy (higher = less human work)
    y = p['overall_routing_accuracy']  # routing accuracy
    ax.scatter(x, y, s=120, color=color, marker=marker, zorder=5, label=name)
    ax.annotate(
        name,
        (x, y),
        textcoords='offset points',
        xytext=(8, 4),
        fontsize=9,
        color=color,
    )

ax.set_xlabel('Auto-Approval Rate (↑ = less accountant work)')
ax.set_ylabel('Overall Routing Accuracy')
ax.set_title('Accuracy vs Workload Reduction Tradeoff')
ax.set_xlim(0.2, 0.95)
ax.set_ylim(0.35, 0.85)
ax.xaxis.set_major_formatter(matplotlib.ticker.PercentFormatter(1.0))
ax.yaxis.set_major_formatter(matplotlib.ticker.PercentFormatter(1.0))
ax.legend(loc='lower right', framealpha=0.9)

plt.tight_layout()
plt.savefig(FIGURES_DIR / 'fig1_accuracy_vs_workload.png', bbox_inches='tight')
plt.show()
print('Saved fig1_accuracy_vs_workload.png')

## Figure 2: Per-Difficulty-Tier Error Rates

In [ ]:
tiers = ['easy', 'medium', 'hard']
tier_labels = ['Easy', 'Medium', 'Hard']
x = np.arange(len(tiers))
n_policies = len(policies)
width = 0.18
offsets = np.linspace(-(n_policies-1)/2 * width, (n_policies-1)/2 * width, n_policies)

fig, ax = plt.subplots(figsize=(8, 5))

for p, name, color, offset in zip(policies, short_names, colors, offsets):
    tier_errs = p.get('error_rate_by_tier', {})
    vals = []
    for tier in tiers:
        v = tier_errs.get(tier)
        vals.append(v if v is not None else 0.0)
    
    bars = ax.bar(x + offset, vals, width, label=name, color=color, alpha=0.85)
    
    # Annotate with N/A where no transactions were auto-approved
    for i, (v, tier) in enumerate(zip(vals, tiers)):
        raw = tier_errs.get(tier)
        if raw is None:
            ax.text(
                x[i] + offset, 0.01, 'N/A',
                ha='center', va='bottom', fontsize=7, color=color,
                rotation=90,
            )

ax.set_xticks(x)
ax.set_xticklabels(tier_labels)
ax.set_xlabel('Transaction Difficulty Tier')
ax.set_ylabel('Error Rate on Auto-Approved Transactions')
ax.set_title('Auto-Approval Error Rate by Difficulty Tier')
ax.yaxis.set_major_formatter(matplotlib.ticker.PercentFormatter(1.0))
ax.set_ylim(0, 0.42)
ax.legend(loc='upper left', framealpha=0.9)
ax.text(
    0.98, 0.95, 'N/A = no transactions\nauto-approved in this tier',
    transform=ax.transAxes, ha='right', va='top',
    fontsize=8, style='italic', color='grey',
)

plt.tight_layout()
plt.savefig(FIGURES_DIR / 'fig2_error_by_tier.png', bbox_inches='tight')
plt.show()
print('Saved fig2_error_by_tier.png')

## Figure 3: Routing Action Distribution

In [ ]:
action_keys = ['AUTO_APPROVE', 'SURFACE_FOR_REVIEW', 'REJECT_FOR_MANUAL']
action_labels = ['Auto-Approve', 'Surface for Review', 'Reject for Manual']
action_colors = ['#4CAF50', '#FF9800', '#F44336']

n = data['n_transactions']
# Build fractions per policy per action
fracs = {ak: [] for ak in action_keys}
for p in policies:
    totals = p.get('action_totals', {})
    for ak in action_keys:
        fracs[ak].append(totals.get(ak, 0) / n)

x = np.arange(len(short_names))
fig, ax = plt.subplots(figsize=(8, 5))

bottoms = np.zeros(len(short_names))
for ak, label, color in zip(action_keys, action_labels, action_colors):
    vals = np.array(fracs[ak])
    bars = ax.bar(x, vals, bottom=bottoms, label=label, color=color, alpha=0.85, edgecolor='white', linewidth=0.5)
    # Label segments > 5%
    for i, (v, b) in enumerate(zip(vals, bottoms)):
        if v > 0.05:
            ax.text(i, b + v/2, f'{v:.0%}', ha='center', va='center', fontsize=9, color='white', fontweight='bold')
    bottoms += vals

ax.set_xticks(x)
ax.set_xticklabels(short_names)
ax.set_ylabel('Fraction of Transactions')
ax.set_title('Routing Action Distribution per Policy')
ax.yaxis.set_major_formatter(matplotlib.ticker.PercentFormatter(1.0))
ax.set_ylim(0, 1.05)
ax.legend(loc='upper right', framealpha=0.9)

plt.tight_layout()
plt.savefig(FIGURES_DIR / 'fig3_action_distribution.png', bbox_inches='tight')
plt.show()
print('Saved fig3_action_distribution.png')

## Figure 4: Learning Curves (from Monitor logs)

In [ ]:
import csv

def load_monitor_csv(variant: str) -> tuple[list, list] | None:
    """Load episode rewards from SB3 Monitor CSV."""
    monitor_path = REPO_ROOT / 'experiments' / 'tensorboard' / f'variant_{variant}' / 'monitor.monitor.csv'
    if not monitor_path.exists():
        return None
    rewards, timesteps = [], []
    with open(monitor_path, newline='') as f:
        # First line is a comment header, second is column names
        lines = f.readlines()
    # Skip comment line(s) starting with '#'
    data_lines = [l for l in lines if not l.startswith('#')]
    reader = csv.DictReader(data_lines)
    cumulative_steps = 0
    for row in reader:
        rewards.append(float(row['r']))
        cumulative_steps += int(row['l'])
        timesteps.append(cumulative_steps)
    return timesteps, rewards

def smooth(values: list, window: int = 5) -> np.ndarray:
    """Simple moving average smoothing."""
    if len(values) < window:
        return np.array(values, dtype=float)
    kernel = np.ones(window) / window
    return np.convolve(values, kernel, mode='valid')

fig, ax = plt.subplots(figsize=(8, 5))

variant_colors = {'A': '#2196F3', 'B': '#4CAF50', 'C': '#FF5722'}
plotted = False

for variant, color in variant_colors.items():
    result = load_monitor_csv(variant)
    if result is None:
        print(f'Monitor CSV for variant {variant} not found, skipping.')
        continue
    timesteps, rewards = result
    smoothed = smooth(rewards, window=5)
    ts_trimmed = timesteps[len(timesteps)-len(smoothed):]
    ax.plot(ts_trimmed, smoothed, color=color, label=f'PPO Variant {variant}', linewidth=1.8)
    ax.fill_between(ts_trimmed, smoothed, alpha=0.1, color=color)
    plotted = True

if plotted:
    ax.set_xlabel('Timesteps')
    ax.set_ylabel('Episode Reward (5-ep rolling mean)')
    ax.set_title('Learning Curves — PPO Variants (Training)')
    ax.legend(loc='lower right', framealpha=0.9)
    ax.xaxis.set_major_formatter(matplotlib.ticker.FuncFormatter(lambda x, _: f'{x/1000:.0f}k'))
    plt.tight_layout()
    plt.savefig(FIGURES_DIR / 'fig4_learning_curves.png', bbox_inches='tight')
    plt.show()
    print('Saved fig4_learning_curves.png')
else:
    print('No monitor data found. Run training first.')

## Summary

All four figures saved to `experiments/results/figures/`. See `comparison_report.md` for full interpretation.